In [ ]:
requirements

In [274]:
import overpy
import pandas as pd
import requests
import urllib, json

In [275]:
df_city = pd.read_csv('worldcities.csv')

In [276]:
df_capitals = df_city[df_city['capital'] == "primary"].reset_index(drop=True)

In [277]:
df_capitals.drop(df_capitals.index[206], inplace=True)

In [278]:
import urllib, json
def bbox_city(name):
    """ Returns bbox value based on the city name
    
    Args:
    name(str) - name of the city or settlement
    
    Returns:
    bbox(lst) - list of geographical points in BBOX format
    """
    link = "https://nominatim.openstreetmap.org/search/" + str(name).replace(" ","-") + "?format=geojson&polygon=1&polygon_geojson=1&limit=1"
    response = urllib.request.urlopen(link)
    data = json.loads(response.read())
    result = data['features'][0]['bbox']
    return result

In [127]:
print(clean_bbox.__doc__)

Transforms BBOX returned by the bbox_city() to the right order.
    
    Args:
    result - bbox returned by the bbox_city() function
    
    Returns:
    bbox_clean - list in BBOX format in the right order
    


In [279]:
df_capitals['bbox'] = df_capitals['city_ascii'].apply(bbox_city)

KeyboardInterrupt: 

In [179]:
df_capitals.to_csv('capitals_bbox.csv')

In [177]:
def clean_bbox(result):
    """Transforms BBOX returned by the bbox_city() to the right order.
    
    Args:
    result - bbox returned by the bbox_city() function
    
    Returns:
    bbox_clean - list in BBOX format in the right order
    """

    myorder = [1, 0, 3, 2]
    result_clean = [result[i] for i in myorder]
    result_clean = str(result_clean)
    bbox_clean = result_clean.replace("[","")
    bbox_clean = bbox_clean.replace("]","")
    bbox_clean = str(bbox_clean)
    return bbox_clean

In [200]:
df_capitals = pd.read_csv("capitals_bbox.csv")

In [224]:
df_capitals['bbox']

0      '0', '2', '2', '.'
1      '6', '-', '3', '.'
2      '4', '1', '5', '.'
3      '7', '3', '4', '.'
4      '9', '1', '0', '.'
              ...        
201    '2', '-', '.', '6'
202    '2', '1', '0', '.'
203    '3', '4', '8', '.'
204    '3', '3', '5', '.'
205    '8', '3', '8', '.'
Name: bbox, Length: 206, dtype: object

In [178]:
df_capitals['bbox'] = df_capitals['bbox'].apply(clean_bbox)

Trees

In [128]:
import time
def trees(bbox_clean):
    """Returns json with list of trees from Open Street Map
    
    Args:
    bbox_clean
    
    Returns:
    response.json() - response of the server in JSON format
    """
    query = 'https://lz4.overpass-api.de/api/interpreter?data=[out:json];(node["natural"="tree"]('+bbox_clean+'););out body;>;out skel qt;'
    response = requests.get(query)
    print(response.status_code)
    return response.json()

In [469]:
import json
number = 203
name = df_capitals['city_ascii'][number]
data = trees(df_capitals['bbox'][number])
                                    
with open(name+'.json', 'w') as convert_file:
     convert_file.write(json.dumps(data))

200


In [9]:

import pandas as pd
import json
df_capitals = pd.read_csv("capitals_final.csv")
path = "trees_capitals/"
name = number = 100
name = df_capitals['city_ascii'][number]

df = pd.read_json(path+name+'.json',orient='index')
trees = pd.DataFrame(df[0]['elements'])
trees = trees.sort_values(by=['lat','lon'], ascending=True)
sample = trees[0:999].reset_index(drop=True)
sample.head()

,type,id,lat,lon,tags
0,node,1211333992,18.452932,-72.348793,"{'natural': 'tree', 'source': 'Bing; survey'}"
1,node,1211125136,18.457998,-72.351077,"{'natural': 'tree', 'source': 'Bing'}"
2,node,640734182,18.508472,-72.508140,{'natural': 'tree'}
3,node,640720955,18.508578,-72.507749,{'natural': 'tree'}
4,node,640720929,18.508589,-72.507910,{'natural': 'tree'}


In [15]:

import folium
import pandas as pd

import folium.plugins as plugins
 
CITY_COORDINATES = (df_capitals['lat'][number], df_capitals['lng'][number])

# for speed purposes
#MAX_RECORDS = 1000

token = "YOUR_MAPBOX_TOKEN"
tileurl = 'https://api.mapbox.com/v4/mapbox.satellite/{z}/{x}/{y}@2x.png?access_token=' + str(token)

# create empty map zoomed in on the city
map = folium.Map(location=CITY_COORDINATES, zoom_start=12,tiles=tileurl, attr='Mapbox')

# add a marker for every record in the filtered data, use a clustered view
for each in sample[0:MAX_RECORDS].iterrows():
    folium.Marker([each[1]['lat'],each[1]['lon']], tooltip=each[1]['lat']).add_to(map)
                                    
display(map)

NameError: name 'df_capitals' is not defined

In [14]:
pip install folium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 5.7 MB/s eta 0:00:00
  Using cached branca-0.6.0-py3-none-any.whl (24 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
#NEW METHOD FOR CALCULATING BBOX 
lon = 5.011857
lat = 52.349148

calc_1 = 4.965501-4.965418
calc_2 = 52.3233715-52.323340
calc_3 = 4.965501-4.965584
calc_4 = 52.3233715-52.323403

bbox_val1 = lon+calc_1
bbox_val2 = lat+calc_2

bbox_val3 = lon+calc_3
bbox_val4 = lat+calc_4

bbox = [bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4]

In [3]:
#create outside geometry polygon

from shapely.geometry import box
#b = box(4.965418,52.323340,4.965584,52.323403)
b = box(bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4)
points = list(b.exterior.coords)

print(b.wkt)

POLYGON ((5.011774 52.3491795, 5.011774 52.3491165, 5.01194 52.3491165, 5.01194 52.3491795, 5.011774 52.3491795))


In [4]:
import geopandas
json = geopandas.GeoSeries(b).to_json()

In [5]:
geo_file = geopandas.GeoSeries(b)
geo_file.crs = "epsg:4326"
geo_file = geo_file.to_crs("EPSG:3857")
geo_file.to_json()

'{"type": "FeatureCollection", "features": [{"id": "0", "type": "Feature", "properties": {}, "geometry": {"type": "Polygon", "coordinates": [[[557908.1296509678, 6863509.512273038], [557908.1296509678, 6863498.031304486], [557926.6086864396, 6863498.031304486], [557926.6086864396, 6863509.512273038], [557908.1296509678, 6863509.512273038]]]}, "bbox": [557908.1296509678, 6863498.031304486, 557926.6086864396, 6863509.512273038]}], "bbox": [557908.1296509678, 6863498.031304486, 557926.6086864396, 6863509.512273038]}'

In [ ]:
geo_file.to_file("testdata.geojson")

In [14]:
def create_bbox(row):
  lon = row['lon']
  lat = row['lat']
  
  calc_1 = 4.965501-4.965418
  calc_2 = 52.3233715-52.323340
  calc_3 = 4.965501-4.965584
  calc_4 = 52.3233715-52.323403

  bbox_val1 = lon+calc_1
  bbox_val2 = lat+calc_2
  bbox_val3 = lon+calc_3
  bbox_val4 = lat+calc_4

  bbox = [bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4]
  b = box(bbox_val1,bbox_val2,bbox_val3
            ,bbox_val4)
  points = list(b.exterior.coords)
  result = geopandas.GeoSeries(b)
  return result


In [30]:
sample['polygon'] = sample.apply(create_bbox, axis=1)
geo_df = sample[['id','polygon']]
import geopandas
gdf = geopandas.GeoDataFrame(
    geo_df, geometry = "polygon")
gdf.crs = "epsg:4326"
gdf = gdf.to_crs("EPSG:3857")
#api_call_data = gdf.to_json()
gdfjson = gdf.to_json()

In [34]:
with open('testapi.json', 'w') as convert_file:
     convert_file.write(json.dumps(gdfjson))

In [10]:
!pip install oauthlib
!pip install requests_oauthlib
!pip install sentinelhub

  Using cached sentinelhub-3.8.0.tar.gz (218 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached aenum-3.1.11-py3-none-any.whl (131 kB)
  Using cached utm-0.7.0.tar.gz (8.7 kB)
  Preparing metadata (setup.py) ... done
  Using cached dataclasses_json-0.5.7-py3-none-any.whl (25 kB)
  Using cached marshmallow_enum-1.5.1-py2.py3-none-any.whl (4.2 kB)
  Using cached typing_inspect-0.8.0-py3-none-any.whl (8.7 kB)
  Using cached marshmallow-3.19.0-py3-none-any.whl (49 kB)
  Created wheel for sentinelhub: filename=sentinelhub-3.8.0-py3-none-any.whl size=242429 sha256=dfc4dc284267652d9a3f069a2cbb9ba12ac2f0938f72f0e6dc183c3d168596cf
  Stored in directory: /Users/jbms/Library/Caches/pip/wheels/aa/b5/d6/7087b2bd8da3b443d043fc7fcc92b69bf0c51614e609c7c6e2
  Created wheel for utm: filename=utm-0.7.0-py3-none-any.whl size=6086 sha256=6547ec2b27634e2a360600f6905d9e72d427a53d8bc7f1ccaa4fb427624a02d4
  S

In [143]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session


from sentinelhub import SHConfig

config = SHConfig()

# Your client credentials
client_id = '96acbb95-6c77-4d93-aaf3-ca218727e1b4'
client_secret = 'F@,ZbT*8PTZ(F}:7@.|,*;#K/K/Ev+#)uRLQ,^t2'

# Create a session
client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

# Get token for the session
token = oauth.fetch_token(token_url='https://services.sentinel-hub.com/oauth/token',
                          client_secret=client_secret)

# All requests using this session will have an access token automatically added
resp = oauth.get("https://services.sentinel-hub.com/oauth/tokeninfo")
print(resp.content)

b'{"sub":"f71bef76-9296-41ab-90ac-459ffdc5f3ac","aud":"96acbb95-6c77-4d93-aaf3-ca218727e1b4","jti":"563aa2f3-0d4e-470b-88ba-2b95752f4b7e","exp":1670862604,"name":"Jakub Supera","email":"jakub.supera@gmail.com","given_name":"Jakub","family_name":"Supera","sid":"0c26c1f9-5eca-4b9d-aba7-3ce25bbd43d4","did":1,"aid":"1f45fa8a-dd28-4373-bae1-3ea4c83e1c78","d":{"1":{"ra":{"rag":1},"t":12000}},"active":true}'


In [144]:
auth = token['access_token']

In [282]:
print(auth)

eyJraWQiOiJzaCIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiJmNzFiZWY3Ni05Mjk2LTQxYWItOTBhYy00NTlmZmRjNWYzYWMiLCJhdWQiOiI5NmFjYmI5NS02Yzc3LTRkOTMtYWFmMy1jYTIxODcyN2UxYjQiLCJqdGkiOiI1NjNhYTJmMy0wZDRlLTQ3MGItODhiYS0yYjk1NzUyZjRiN2UiLCJleHAiOjE2NzA4NjI2MDQsIm5hbWUiOiJKYWt1YiBTdXBlcmEiLCJlbWFpbCI6Impha3ViLnN1cGVyYUBnbWFpbC5jb20iLCJnaXZlbl9uYW1lIjoiSmFrdWIiLCJmYW1pbHlfbmFtZSI6IlN1cGVyYSIsInNpZCI6IjBjMjZjMWY5LTVlY2EtNGI5ZC1hYmE3LTNjZTI1YmJkNDNkNCIsImRpZCI6MSwiYWlkIjoiMWY0NWZhOGEtZGQyOC00MzczLWJhZTEtM2VhNGM4M2UxYzc4IiwiZCI6eyIxIjp7InJhIjp7InJhZyI6MX0sInQiOjEyMDAwfX19.ol9htSUopdNfLzp7PmIcsge0s104Su5ai9uGlLPjghnAHX-y8fJYvv0xGK5e2miwI91Hv6lehZ7KzTDpOpGNam0Mykq65YTzP8z_ncmge-eXfMR460MwGH2Hcb3fQAphNilkOoKrbWIsoLxcN6uHA6-l7_F_PRzYWH6jyxm343hIBDJmMN7Y01jPW6QaeJic0FHQDtmoIYaywk-FzBc0ozY8h-dDottRu8wlJGXLvurL2gScEjA8n-o9n4WEH9NIvm2p4hMv9wjyHQU1y7GTdrKkHn4TBtBdgPpOw_bkuz2BY0EH1tXQpSSv8uQSQ2UCEUh4Q5MfHJsDff6uuHiBrg


In [3]:

def sentinel_authentication ():
    '''Requests authentication token from Sentinel Hub

    Returns:
    auth(str) - access_token '''
    from oauthlib.oauth2 import BackendApplicationClient
    from requests_oauthlib import OAuth2Session
    from sentinelhub import SHConfig

    config = SHConfig()

    # Your client credentials
    client_id = '96acbb95-6c77-4d93-aaf3-ca218727e1b4'
    client_secret = 'F@,ZbT*8PTZ(F}:7@.|,*;#K/K/Ev+#)uRLQ,^t2'

    # Create a session
    client = BackendApplicationClient(client_id=client_id)
    oauth = OAuth2Session(client=client)

    # Get token for the session
    token = oauth.fetch_token(token_url='https://services.sentinel-hub.com/oauth/token',
                              client_secret=client_secret)

    # All requests using this session will have an access token automatically added
    resp = oauth.get("https://services.sentinel-hub.com/oauth/tokeninfo")
    
    auth = token['access_token']
    
    return auth

In [50]:
#######restart work here ####


import requests, json

def geotiff_caller():
    
    headers = {
        'Authorization': 'Bearer ' +sentinel_authentication() ,
    }

    files = {'request': (None, '{"input": {"bounds": {"properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}, "bbox": [139.368439, 35.515461, 140.060577, 35.877924]}, "data": [{"type": "sentinel-2-l2a", "dataFilter": {"timeRange": {"from": "2022-10-21T01:27:21Z", "to": "2022-10-21T01:27:21Z"}, "maxCloudCoverage": 2}, "processing": {"harmonizeValues": "true"}}]}, "output": {"width": 512, "height": 512, "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]}}'), 'evalscript': (None, '//VERSION=3\nfunction setup() {\n  return{\n    input: [{\n      bands: ["B04", "B08"],\n      units: "REFLECTANCE"\n    }],\n    output: {\n      id: "default",\n      bands: 1,\n      sampleType: SampleType.FLOAT32\n    }\n  }\n}\n\nfunction evaluatePixel(sample) {\n  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04)\n  return [ ndvi ]\n}')}
    
    request = json.loads(files["request"][1])
    request["input"]["bounds"]["bbox"] = [139.368439,35.515461,140.060577,35.877924]
    request["input"]["data"][0]["dataFilter"]["timeRange"]["from"] = '2022-02-27T00:00:00Z'
    request["input"]["data"][0]["dataFilter"]["timeRange"]["to"] = '2022-03-06T00:00:00Z'
    request["input"]["data"][0]["dataFilter"]["maxCloudCoverage"] = 15
    files["request"] = tuple( (None, json.dumps(request)) )
    print(files)
    

    #json.loads(files["request"][1])["input"]["bounds"]["bbox"] = '[135.8536855, 20.2145811, 154.205541, 35.8984245]'
    #json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["timeRange"]["from"] = ["properties.datetime"]
    #json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["timeRange"]["to"] = ["properties.datetime"]
    #json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["maxCloudCoverage"] = 2

    response = requests.post('https://services.sentinel-hub.com/api/v1/process', headers=headers, files=files)
    with open("tokyo_test1.tif", "wb") as f:
        f.write(response.content)
        
   
    return response

In [51]:
test_response  = geotiff_caller()

{'request': (None, '{"input": {"bounds": {"properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}, "bbox": [139.368439, 35.515461, 140.060577, 35.877924]}, "data": [{"type": "sentinel-2-l2a", "dataFilter": {"timeRange": {"from": "2022-02-27T00:00:00Z", "to": "2022-03-06T00:00:00Z"}, "maxCloudCoverage": 15}, "processing": {"harmonizeValues": "true"}}]}, "output": {"width": 512, "height": 512, "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]}}'), 'evalscript': (None, '//VERSION=3\nfunction setup() {\n  return{\n    input: [{\n      bands: ["B04", "B08"],\n      units: "REFLECTANCE"\n    }],\n    output: {\n      id: "default",\n      bands: 1,\n      sampleType: SampleType.FLOAT32\n    }\n  }\n}\n\nfunction evaluatePixel(sample) {\n  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04)\n  return [ ndvi ]\n}')}


In [41]:
test_response.ok

True

In [3]:
import os
metadata = os.popen('gdalinfo /Users/jbms/TI/Tokyo/default.tif').read()
print(metadata)

Driver: GTiff/GeoTIFF
Files: /Users/jbms/TI/Tokyo/default.tif
Size is 512, 343
Coordinate System is:
GEOGCRS["WGS 84",
    DATUM["World Geodetic System 1984",
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
    CS[ellipsoidal,2],
        AXIS["geodetic latitude (Lat)",north,
            ORDER[1],
            ANGLEUNIT["degree",0.0174532925199433]],
        AXIS["geodetic longitude (Lon)",east,
            ORDER[2],
            ANGLEUNIT["degree",0.0174532925199433]],
    USAGE[
        SCOPE["unknown"],
        AREA["World"],
        BBOX[-90,-180,90,180]],
    ID["EPSG",4326]]
Data axis to CRS axis mapping: 2,1
Origin = (12.446930000000000,41.917096000000001)
Pixel Size = (0.000183732421875,-0.000137096209913)
Metadata:
  AREA_OR_POINT=Area
  TIFFTAG_RESOLUTIONUNIT=1 (unitless)
  TIFFTAG_XRESOLUTION=1
  TIFFTAG_YRESOLUTION=1
Image Structure Metadata:
  COMPRESSION=DEFLATE


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


In [18]:
with open("tokyo_test1.tif", "wb") as f:
    f.write(response.content)

NameError: name 'response' is not defined

In [4]:
files = {
        'request': (None, '{\n    "input": {\n        "bounds": {\n            "properties": {\n                "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"\n            },\n            "bbox": [\n                4.850807,52.302391,5.047874,52.387963\n            ]\n        },\n        "data": [\n            {\n                "type": "sentinel-2-l2a",\n                "dataFilter": {\n                    "timeRange": {\n                        "from": "2020-04-20T10:56:23Z",\n                        "to": "2020-04-20T10:56:23Z"\n                    },\n                    "maxCloudCoverage": 1\n                },\n                "processing": {\n                    "harmonizeValues": "true"       \n                }\n            }\n        ]\n    },\n    "output": {\n        "width": 512,\n        "height": 512,\n        "responses": [\n            {\n                "identifier": "default",\n                "format": {\n                    "type": "image/tiff"\n                }\n            }\n        ]\n    }\n    \n}'),
        'evalscript': (None, '//VERSION=3\nfunction setup() {\n  return{\n    input: [{\n      bands: ["B04", "B08"],\n      units: "REFLECTANCE"\n    }],\n    output: {\n      id: "default",\n      bands: 1,\n      sampleType: SampleType.FLOAT32\n    }\n  }\n}\n\nfunction evaluatePixel(sample) {\n  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04)\n  return [ ndvi ]\n}'),
    }

json.loads(files["request"][1])["input"]["bounds"]["bbox"] = '[139.368439,35.515461,140.060577,35.877924]'
json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["timeRange"]["from"] = str(["properties.datetime"])
json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["timeRange"]["to"] = str(["properties.datetime"])
json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["maxCloudCoverage"] = str(2)

NameError: name 'json' is not defined

In [5]:
request = json.loads(files["request"][1])
request["input"]["bounds"]["bbox"] = [135.8536855, 20.2145811, 154.205541, 35.8984245]
request

NameError: name 'json' is not defined

In [438]:
files["request"] = tuple( (None, request) )

In [650]:
city_dates = pd.read_csv("tokyo_dates.csv")

In [660]:
city_dates["properties.datetime"]

0      2022-10-21T01:27:21Z
1      2022-10-21T01:27:18Z
2      2022-10-18T01:20:05Z
3      2022-10-18T01:19:58Z
4      2022-10-18T01:19:53Z
               ...         
195    2022-09-01T01:28:46Z
196    2022-09-01T01:28:39Z
197    2022-09-01T01:28:35Z
198    2022-09-01T01:28:32Z
199    2022-09-01T01:28:24Z
Name: properties.datetime, Length: 200, dtype: object

In [17]:
response = geotiff_caller(city_dates["properties.datetime"][0])
"""Writes geotiff answer from API call as file"""

#with open("tokyo_test.tif", "wb") as f:
#   f.write(response.content)

NameError: name 'city_dates' is not defined

In [16]:
with open("tokyo_test1.tif", "wb") as f:
    f.write(response.content)

NameError: name 'response' is not defined

In [681]:
response.status_code

200

In [48]:
string = {'request': (None, '{"input": {"bounds": {"properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}, "bbox": [139.368439, 35.515461, 140.060577, 35.877924]}, "data": [{"type": "sentinel-2-l2a", "dataFilter": {"timeRange": {"from": "2020-04-20T00:00:01Z", "to": "2020-04-20T23:59:59Z"}, "maxCloudCoverage": 2}, "processing": {"harmonizeValues": "true"}}]}, "output": {"width": 512, "height": 512, "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]}}'), 'evalscript': (None, '//VERSION=3\nfunction setup() {\n  return{\n    input: [{\n      bands: ["B04", "B08"],\n      units: "REFLECTANCE"\n    }],\n    output: {\n      id: "default",\n      bands: 1,\n      sampleType: SampleType.FLOAT32\n    }\n  }\n}\n\nfunction evaluatePixel(sample) {\n  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04)\n  return [ ndvi ]\n}')}

In [ ]:
string

{'request': (None,
  '{"input": {"bounds": {"properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}, "bbox": [139.368439, 35.515461, 140.060577, 35.877924]}, "data": [{"type": "sentinel-2-l2a", "dataFilter": {"timeRange": {"from": "2020-04-20T00:00:01Z", "to": "2020-04-20T23:59:59Z"}, "maxCloudCoverage": 2}, "processing": {"harmonizeValues": "true"}}]}, "output": {"width": 512, "height": 512, "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]}}'),
 'evalscript': (None,
  '//VERSION=3\nfunction setup() {\n  return{\n    input: [{\n      bands: ["B04", "B08"],\n      units: "REFLECTANCE"\n    }],\n    output: {\n      id: "default",\n      bands: 1,\n      sampleType: SampleType.FLOAT32\n    }\n  }\n}\n\nfunction evaluatePixel(sample) {\n  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04)\n  return [ ndvi ]\n}')}

In [540]:
str(df["properties.datetime"][0])

NameError: name 'df' is not defined

In [338]:
bbox = json.loads(files["request"][1])["input"]["bounds"]["bbox"]
date_from = json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["timeRange"]["from"]
date_to = json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["timeRange"]["to"]
cloud_coverage = json.loads(files["request"][1])["input"]["data"][0]["dataFilter"]["maxCloudCoverage"]

In [539]:
json.loads(files["request"][1])["input"]["bounds"]["bbox"]

IndexError: tuple index out of range

In [281]:
"""Returns dates on which cloud cover was minimal

Args:
bbox - expects it to be in format lng/lat
json_data - based of json_data_edit which can be modified
"""


import requests
def check_data (json_data):
        headers = {
        'Authorization': 'Bearer ' +auth ,
        'Content-Type': 'application/json',
        }

        json_data = json_data_edit

        response = requests.post('https://services.sentinel-hub.com/api/v1/catalog/1.0.0/search', headers=headers, json=json_data)
        
        return response.json()
    
# Note: json_data will not be serialized by requests
# exactly as it was in the original request.
#data = '{\n      "bbox": [13,45,14,46],\n      "datetime": "2019-12-10T00:00:00Z/2019-12-10T23:59:59Z",\n      "collections": ["sentinel-2-l2a"],\n      "limit": 5,\n      "next": 5,\n"filter": "eo:cloud_cover > 90"\n  }'
#response = requests.post('https://services.sentinel-hub.com/api/v1/catalog/1.0.0/search', headers=headers, data=data)

In [268]:
json_data_edit = {
    'bbox': [
        4.850807,52.302391,5.047874,52.387963
    ],
    
    'datetime': '2018-01-01T00:00:00Z/2022-12-31T23:59:59Z',
    'collections': [
        'sentinel-2-l2a',
    ],
    'limit': 100,
    'next': 100,
    'filter': 'eo:cloud_cover < 5',
}

In [252]:
json_data_edit["bbox"]

[4.850807, 52.302391, 5.047874, 52.387963]
2018-01-01T00:00:00Z/2022-12-31T23:59:59Z
eo:cloud_cover < 5 


In [270]:
response.json()["context"]

{'limit': 100, 'returned': 89}

In [292]:
import requests

response = requests.post('https://services.sentinel-hub.com/api/v1/process',
headers={"Authorization" : "Bearer "+auth,},
json= json_text
                        )

In [113]:
json_text={
    "input": {
        "bounds": {
            "bbox": [
                13.822174072265625,
                45.85080395917834,
                14.55963134765625,
                46.29191774991382
            ]
        },
        "data": [{
            "type": "sentinel-2-l2a"
        }]
    },
    "evalscript": """
    //VERSION=3

    function setup() {
      return {
        input: ["B02", "B03", "B04"],
        output: {
          bands: 3
        }
      };
    }

    function evaluatePixel(
      sample,
      scenes,
      inputMetadata,
      customData,
      outputMetadata
    ) {
      return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02];
    }
    """
}

In [228]:
import pandas as pd
data_bbox = pd.read_csv("capitals_final.csv")

In [259]:
result = data_bbox['bbox'][0]
result = result.split(",")

In [260]:
myorder = [1, 0, 3, 2]
result_clean = [result[i] for i in myorder]



In [279]:
result_clean

['135.8536855', '20.2145811', '154.205541', '35.8984245']

In [255]:
str(result.split(",")).replace("'","")

AttributeError: 'list' object has no attribute 'split'

In [226]:
data_bbox['bbox'] = data_bbox['bbox'].apply(clean_bbox)

In [290]:
json_text['input']['bounds']['bbox'] = result

In [291]:
json_text

{'input': {'bounds': {'bbox': ['20.2145811',
    '135.8536855',
    '35.8984245',
    '154.205541']},
  'data': [{'type': 'sentinel-2-l2a'}]},
 'evalscript': '\n    //VERSION=3\n\n    function setup() {\n      return {\n        input: ["B02", "B03", "B04"],\n        output: {\n          bands: 3\n        }\n      };\n    }\n\n    function evaluatePixel(\n      sample,\n      scenes,\n      inputMetadata,\n      customData,\n      outputMetadata\n    ) {\n      return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02];\n    }\n    '}

In [58]:
curl_text =  {
        "bounds": {
            "properties": {
                "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [
                            -94.04798984527588,
                            41.7930725281021
                        ],
                        [
                            -94.04803276062012,
                            41.805773608962869
                        ],
                        [
                            -94.06738758087158,
                            41.805901566741308
                        ],
                        [
                            -94.06734466552735,
                            41.7967199475024
                        ],
                        [
                            -94.06223773956299,
                            41.79144072064381
                        ],
                        [
                            -94.0504789352417,
                            41.791376727347969
                        ],
                        [
                            -94.05039310455322,
                            41.7930725281021
                        ],
                        [
                            -94.04798984527588,
                            41.7930725281021
                        ]
                    ]
                ]
            }
        },
        "data": [
            {
                "type": "sentinel-2-l2a",
                "dataFilter": {
                    "timeRange": {
                        "from": "2018-10-01T00:00:00Z",
                        "to": "2018-12-20T00:00:00Z"
                    }
                },
                "processing": {
                    "harmonizeValues": "true"       
                }
            }
        ]
    }

In [86]:
"""Assigns new values to cooridnates, timeRange - from and to """

curl_text["bounds"]["geometry"]["coordinates"] = 'test'
curl_text["data"][0]["dataFilter"]["timeRange"]["from"] = 'test1'
curl_text["data"][0]["dataFilter"]["timeRange"]["to"] = 'test2'

'2018-10-01T00:00:00Z'

In [254]:
import session_info
session_info.show()